# Lesson 3: Loading Data into PostgreSQL

**Week 4 · Data Engineering Course**

---

CSV files are fine for small datasets and one-off analysis, but production pipelines store data in a **database**. This lesson shows you how to:

- Connect to PostgreSQL from Python
- Create a table with the right column types
- Insert rows efficiently
- Use an **upsert** to safely re-run a pipeline without creating duplicates
- Query the data back with `pd.read_sql()`

**Install before starting:**
```
pip install psycopg2-binary
```

**You need PostgreSQL running.** If you installed it in Week 1, start it with pgAdmin or from the command line. Create a database called `weather_db` in pgAdmin before running this notebook.

In [ ]:
import psycopg2
import pandas as pd
from pathlib import Path

DATA  = Path('data')
CLEAN = DATA / 'clean'

print(f'psycopg2 version: {psycopg2.__version__}')

---

## 3.1 Connecting to PostgreSQL

`psycopg2.connect()` opens a connection to PostgreSQL. You give it the same credentials you use in pgAdmin.

In [ ]:
# Replace these values with your actual PostgreSQL credentials
DB_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'dbname':   'weather_db',
    'user':     'postgres',
    'password': 'your_password_here',   # change this
}

try:
    conn = psycopg2.connect(**DB_CONFIG)
    print('Connected to PostgreSQL!')
    print(f'Server version: {conn.server_version}')
except psycopg2.OperationalError as e:
    print(f'Connection failed: {e}')
    print('Check that PostgreSQL is running and your credentials are correct.')

### Connection concepts

- A **connection** (`conn`) is the link between Python and PostgreSQL.
- A **cursor** (`cur`) is used to send SQL commands and read results — like a handle on a specific query.
- You must **commit** a transaction to save changes, or **rollback** to undo them.
- Always **close** connections when you are done, or use a `with` block.

---

## 3.2 Creating a Table

Before inserting any data you need to define the table structure. This is the same SQL you learned in Week 1 and 2 — you just run it from Python.

In [ ]:
# CREATE TABLE IF NOT EXISTS is safe to re-run — it does nothing if the table already exists
create_sql = '''
    CREATE TABLE IF NOT EXISTS weather_daily (
        id           SERIAL PRIMARY KEY,
        city         VARCHAR(100) NOT NULL,
        date         DATE         NOT NULL,
        temp_max_c   NUMERIC(5,2),
        temp_min_c   NUMERIC(5,2),
        rain_mm      NUMERIC(7,2),
        wind_max_kmh NUMERIC(6,2),
        latitude     NUMERIC(8,4),
        longitude    NUMERIC(8,4),
        timezone     VARCHAR(50),
        UNIQUE (city, date)   -- no duplicate city+date combinations
    );
'''

# cursor is used to send SQL commands
with conn.cursor() as cur:
    cur.execute(create_sql)
conn.commit()   # save the change

print('Table weather_daily created (or already exists).')

In [ ]:
# Verify the table exists by describing its columns
with conn.cursor() as cur:
    cur.execute('''
        SELECT column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_name = 'weather_daily'
        ORDER BY ordinal_position;
    ''')
    rows = cur.fetchall()

print(f'{"Column":<20}  {"Type":<20}  {"Nullable"}')
print('-' * 55)
for col, dtype, nullable in rows:
    print(f'{col:<20}  {dtype:<20}  {nullable}')

---

## 3.3 Inserting Rows

Use `cur.executemany()` to insert multiple rows in one call. Always use **parameterised queries** — never build SQL strings by concatenating user data. Parameterised queries prevent SQL injection.

In [ ]:
# Load the clean weather CSV from Lesson 2
try:
    weather = pd.read_csv(CLEAN / 'weather_forecast.csv', parse_dates=['date'])
    print(f'Loaded {len(weather)} rows')
except FileNotFoundError:
    print('Run Lesson 2 first to generate weather_forecast.csv')

In [ ]:
# Convert the DataFrame to a list of tuples — one tuple per row
# The column order must match the INSERT statement below
columns = ['city', 'date', 'temp_max_c', 'temp_min_c', 'rain_mm',
           'wind_max_kmh', 'latitude', 'longitude', 'timezone']

rows_to_insert = [
    tuple(row)
    for row in weather[columns].itertuples(index=False)
]

print(f'Rows to insert: {len(rows_to_insert)}')
print('First row:', rows_to_insert[0])

In [ ]:
# INSERT with parameterised placeholders
# %s is the psycopg2 placeholder — it gets replaced safely by the values
insert_sql = '''
    INSERT INTO weather_daily
        (city, date, temp_max_c, temp_min_c, rain_mm, wind_max_kmh,
         latitude, longitude, timezone)
    VALUES
        (%s, %s, %s, %s, %s, %s, %s, %s, %s);
'''

with conn.cursor() as cur:
    cur.executemany(insert_sql, rows_to_insert)
conn.commit()

print(f'Inserted {len(rows_to_insert)} rows into weather_daily')

---

## 3.4 Querying Data Back

`pd.read_sql()` runs a SQL query and returns the result as a DataFrame — perfect for analysis.

In [ ]:
# Read everything back
df_db = pd.read_sql('SELECT * FROM weather_daily ORDER BY city, date;', conn)
print(f'Rows in database: {len(df_db)}')
df_db.head(10)

In [ ]:
# Run analytical queries directly in SQL
hottest_days = pd.read_sql('''
    SELECT city, date, temp_max_c
    FROM weather_daily
    ORDER BY temp_max_c DESC
    LIMIT 5;
''', conn)

print('Hottest days in the database:')
print(hottest_days)

In [ ]:
# Aggregate by city in SQL
summary = pd.read_sql('''
    SELECT
        city,
        COUNT(*)              AS days,
        ROUND(AVG(temp_max_c), 1) AS avg_max_temp,
        ROUND(SUM(rain_mm), 1)    AS total_rain_mm
    FROM weather_daily
    GROUP BY city
    ORDER BY avg_max_temp DESC;
''', conn)

print(summary)

---

## 3.5 Upsert — Safe Re-runs

A pipeline often runs on a schedule — daily or hourly. If you run `INSERT` again on data you already loaded, you will get duplicate rows.

The solution is an **upsert** (PostgreSQL calls it `INSERT ... ON CONFLICT DO UPDATE`): insert the row, but if a row with the same unique key already exists, update it instead of inserting a second copy.

In [ ]:
# Upsert: safe to run multiple times
# 'ON CONFLICT (city, date)' matches our UNIQUE constraint
# 'DO UPDATE SET ...' overwrites the existing row with fresh values
upsert_sql = '''
    INSERT INTO weather_daily
        (city, date, temp_max_c, temp_min_c, rain_mm, wind_max_kmh,
         latitude, longitude, timezone)
    VALUES
        (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (city, date) DO UPDATE SET
        temp_max_c   = EXCLUDED.temp_max_c,
        temp_min_c   = EXCLUDED.temp_min_c,
        rain_mm      = EXCLUDED.rain_mm,
        wind_max_kmh = EXCLUDED.wind_max_kmh;
'''

with conn.cursor() as cur:
    cur.executemany(upsert_sql, rows_to_insert)
conn.commit()

# Count rows — should still be the same as before
with conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM weather_daily;')
    count = cur.fetchone()[0]

print(f'Rows after upsert: {count}  (no duplicates added)')

In [ ]:
# Wrapping it all together: a load_to_postgres() function

def load_to_postgres(df, conn, table='weather_daily'):
    '''Upsert a weather DataFrame into PostgreSQL. Returns the number of rows processed.'''
    columns = ['city', 'date', 'temp_max_c', 'temp_min_c', 'rain_mm',
               'wind_max_kmh', 'latitude', 'longitude', 'timezone']

    rows = [tuple(r) for r in df[columns].itertuples(index=False)]

    sql = f'''
        INSERT INTO {table}
            (city, date, temp_max_c, temp_min_c, rain_mm, wind_max_kmh,
             latitude, longitude, timezone)
        VALUES
            (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (city, date) DO UPDATE SET
            temp_max_c   = EXCLUDED.temp_max_c,
            temp_min_c   = EXCLUDED.temp_min_c,
            rain_mm      = EXCLUDED.rain_mm,
            wind_max_kmh = EXCLUDED.wind_max_kmh;
    '''

    with conn.cursor() as cur:
        cur.executemany(sql, rows)
    conn.commit()
    return len(rows)

n = load_to_postgres(weather, conn)
print(f'Upserted {n} rows.')

---

## 3.6 Cleaning Up

In [ ]:
# Close the connection when you are done
# In a long-running pipeline you would manage one connection per script run
conn.close()
print('Connection closed.')

In [ ]:
# To drop the table and start fresh during development:
# (only run this when you want to reset)
#
# conn = psycopg2.connect(**DB_CONFIG)
# with conn.cursor() as cur:
#     cur.execute('DROP TABLE IF EXISTS weather_daily;')
# conn.commit()
# print('Table dropped.')

print('(Drop code is commented out — uncomment to reset the table.)')

---

## Key Takeaways

1. `psycopg2.connect(**config)` opens a connection to PostgreSQL. A **cursor** is used to execute SQL; a **connection** tracks the transaction.
2. Always call `conn.commit()` after INSERT/UPDATE/DELETE to save changes to the database.
3. **Parameterised queries** use `%s` as placeholders. psycopg2 fills them in safely — never build SQL strings by concatenating values, which would expose you to SQL injection.
4. `cur.executemany(sql, list_of_tuples)` inserts many rows efficiently in one call.
5. `pd.read_sql(sql, conn)` runs a SQL query and returns a DataFrame — the cleanest way to read from PostgreSQL into pandas for analysis.
6. **Upsert** (`INSERT ... ON CONFLICT DO UPDATE`) is how you make a pipeline safe to re-run. It inserts new rows and updates existing ones instead of creating duplicates. The `UNIQUE` constraint on the table is what enables this.
7. Close the connection with `conn.close()` when you are done, or wrap the whole thing in a `with psycopg2.connect(...) as conn:` block for automatic cleanup.